# Pre-label bằng SAM3 (text-prompt) trên Colab — xuất COCO JSON để import vào CVAT

**Mục đích:** thay vì vẽ box tay từ đầu cho toàn bộ ảnh, dùng SAM3 (model segmentation mạnh nhất hiện có, hỗ trợ prompt bằng text như "helmet", "license plate"...) để tự động khoanh vùng object trước — bạn chỉ cần **sửa lại** trong CVAT (xoá box sai, sửa box lệch, gán đúng label) thay vì vẽ từ đầu.

**Giới hạn cần biết trước:**
- SAM3 khoanh được vùng "chân người" nhưng KHÔNG tự phân biệt được đang lái (`Leg Riding`) hay đang dắt bộ (`Leg Standing`) — notebook này gán mặc định tất cả là `Leg Riding`, bạn cần tự đổi label những case dắt bộ trong CVAT sau khi import.
- `No Plate` (xe không gắn biển) KHÔNG auto-label được vì đây là sự vắng mặt của object — cần tự đánh dấu tay hoàn toàn trong CVAT.

**Trước khi chạy (bắt buộc):**
1. **Runtime > Change runtime type > GPU** (T4 free tier là đủ).
2. Trên máy bạn, nén 3 thư mục thành 3 file zip riêng: `datasets/cvat_review/helmet_review`, `plate_review`, `new_classes_pool` (nén NGUYÊN NỘI DUNG thư mục — mở zip ra phải thấy trực tiếp các file `.jpg`, không có thư mục con lồng bên trong).
3. Điền `HF_TOKEN` ở ô config bên dưới (xem hướng dẫn lấy token cuối cell này) — `SAM3_MODEL_ID` đã điền sẵn `facebook/sam3`.
4. **Chạy cell "fix môi trường" (cell ngay sau ô config) → RESTART RUNTIME khi cell đó bảo → rồi mới chạy tiếp các cell còn lại (không phải 1 lần "Run all" duy nhất — pip cài numpy/scipy mới bắt buộc phải restart runtime mới áp dụng được, đây không phải lỗi, là quy trình Colab luôn cần khi đổi version numpy).** Đến ô "Upload ảnh" sẽ có nút Choose Files — chọn đúng file zip tương ứng từng lần.

**Kết quả:** 3 file COCO JSON tự động tải về máy bạn qua `files.download(...)` ở cell cuối — import từng file vào đúng CVAT task tương ứng (Task → Actions → Upload annotations → format **COCO 1.0**).

**Lấy HF Token:** vào [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → **Create new token** → loại **Read** → copy dán vào `HF_TOKEN`.

**Nếu lỗi:** báo lại traceback đầy đủ (không chỉ dòng cuối) để chẩn đoán đúng chỗ. Nếu vừa cài package mới mà lỗi lạ (vd liên quan `numpy`), dùng **Runtime > Restart session** rồi chạy tiếp từ cell config trở đi (không chạy lại cell "fix môi trường" trừ khi lỗi quay lại) — pip cài xong không tự áp dụng cho module đã import sẵn trong runtime đang chạy.

In [ ]:
CONF_THRESHOLD = 0.25  # ngưỡng confidence tối thiểu giữ lại 1 detection — để thấp (pre-label
                        # chỉ cần "gợi ý", lọc/xoá box sai trong CVAT còn nhanh hơn bỏ sót)

# Dán token HF của bạn (huggingface.co/settings/tokens, tạo loại "Read").
HF_TOKEN = ""  # BẠN tự điền — vd: "hf_xxxxxxxxxxxxxxxxxxxx"

# facebook/sam3 — đã tra chính thức trên huggingface.co/docs/transformers/model_doc/sam3.
SAM3_MODEL_ID = "facebook/sam3"

In [ ]:
# Dùng HuggingFace transformers — quản lý dependency tốt hơn nhiều so với cài SAM3 trực
# tiếp từ repo git của Meta (repo git tự ép numpy<2, làm vỡ torch/torchvision/opencv có
# sẵn trong Colab, lỗi "numpy.dtype size changed").
#
# LỖI THẬT ĐÃ GẶP: "NumPy / Numba binary conflict" — Colab cài sẵn numba biên dịch khớp
# với numpy gốc của máy ảo; khi ta nâng numpy lên bản mới hơn cho transformers/SAM3, numba
# cũ không còn đọc được ABI numpy mới → lỗi ngay cả khi notebook này không hề import numba
# trực tiếp (gói khác âm thầm import nó). Fix: GỠ HẲN numba (không dùng tới) thay vì cố ép
# version khớp nhau, rồi cài numpy+scipy CÙNG LÚC trong 1 lệnh để pip tự chọn 1 cặp ABI khớp
# (cài numpy rồi cài scipy riêng — 2 lệnh tách nhau — không đảm bảo pip khớp ABI, đã dính
# lỗi kiểu "dtype size changed"/"cannot import _slice"/"ufunc no __module__" vì lý do này).
#
# CHỈ 1 CELL DUY NHẤT LÀM VIỆC NÀY — không thêm cell cài lại numpy/scipy ở chỗ khác trong
# notebook (từng có 2 cell làm cùng việc đá nhau, cell sau ghi đè version cell trước chọn).
!pip uninstall -y numba sam3 2>/dev/null || true
!pip install -q -U --force-reinstall numpy scipy
!pip install -q -U transformers accelerate safetensors huggingface_hub

print("Đã cài xong.")
print("BẮT BUỘC: Runtime > Restart session NGAY BÂY GIỜ (pip cài xong không tự áp dụng cho")
print("module đã import sẵn trong runtime đang chạy). Sau khi restart, chạy tiếp TỪ CELL")
print("PHÍA DƯỚI cell này trở đi — không cần Run all lại từ đầu, cell này không cần chạy lại.")

In [ ]:
# --- Load SAM3 qua HuggingFace transformers ---
# API đã verify thật từ huggingface.co/docs/transformers/model_doc/sam3 (không đoán):
# Sam3Model + Sam3Processor, gọi processor(images=.., text=..) rồi post-process bằng
# processor.post_process_instance_segmentation(...) — trả về dict có "boxes" (xyxy, pixel
# tuyệt đối) + "scores". Model dùng ảnh input 1008x1008 mặc định, không cần chỉnh gì thêm.
import torch
from transformers import Sam3Model, Sam3Processor

assert HF_TOKEN, "Điền HF_TOKEN ở ô config phía trên trước."

model = Sam3Model.from_pretrained(SAM3_MODEL_ID, token=HF_TOKEN, device_map="auto")
processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID, token=HF_TOKEN)
model.eval()
print("device:", model.device)

def predict_boxes(image_path: str, text_prompt: str, conf_threshold: float = CONF_THRESHOLD):
    """Trả về list (x1, y1, x2, y2, score) cho 1 text prompt trên 1 ảnh."""
    from PIL import Image
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, text=text_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=conf_threshold,
        mask_threshold=0.5,
        target_sizes=inputs.get("original_sizes").tolist(),
    )[0]
    boxes = []
    for box, score in zip(results["boxes"], results["scores"]):
        x1, y1, x2, y2 = [float(v) for v in box]
        boxes.append((x1, y1, x2, y2, float(score)))
    return boxes

In [ ]:
import os, json, glob, zipfile
from PIL import Image

def build_coco(image_dir: str, prompt_to_label: dict, out_path: str):
    """
    Chạy SAM3 với từng (text_prompt -> label) trên toàn bộ ảnh trong image_dir,
    ghi kết quả ra 1 file COCO JSON (categories = các label trong prompt_to_label).
    """
    categories = [{"id": i + 1, "name": label, "supercategory": "none"}
                  for i, label in enumerate(sorted(set(prompt_to_label.values())))]
    label_to_cat_id = {c["name"]: c["id"] for c in categories}

    images, annotations = [], []
    ann_id = 1
    files_ = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))
    print(f"{image_dir}: {len(files_)} ảnh")
    if not files_:
        raise FileNotFoundError(
            f"Thư mục '{image_dir}' không có file .jpg nào. Kiểm tra lại zip đã giải nén "
            f"đúng chỗ chưa — nội dung hiện có: {glob.glob(os.path.join(image_dir, '*'))[:10]}"
        )

    for img_id, fpath in enumerate(files_, start=1):
        with Image.open(fpath) as im:
            w, h = im.size
        images.append({
            "id": img_id,
            "file_name": os.path.basename(fpath),
            "width": w,
            "height": h,
        })
        for prompt, label in prompt_to_label.items():
            for x1, y1, x2, y2, score in predict_boxes(fpath, prompt):
                annotations.append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": label_to_cat_id[label],
                    "bbox": [x1, y1, x2 - x1, y2 - y1],  # COCO: [x, y, width, height]
                    "area": (x2 - x1) * (y2 - y1),
                    "iscrowd": 0,
                    "score": score,
                })
                ann_id += 1
        if img_id % 20 == 0:
            print(f"  {img_id}/{len(files_)}")

    coco = {"images": images, "annotations": annotations, "categories": categories}
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False)
    print(f"Xong: {out_path} — {len(images)} ảnh, {len(annotations)} box")

def upload_and_unzip(extract_to: str) -> str:
    """Hiện nút Choose Files, giải nén zip vừa upload vào extract_to, trả về path ảnh thật
    (tự dò 1 cấp xuống nếu zip có thư mục con lồng bên trong thay vì để .jpg ở gốc)."""
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(extract_to)
    os.remove(zip_name)
    if glob.glob(os.path.join(extract_to, "*.jpg")):
        return extract_to
    nested = [d for d in glob.glob(os.path.join(extract_to, "*")) if os.path.isdir(d)]
    if len(nested) == 1 and glob.glob(os.path.join(nested[0], "*.jpg")):
        return nested[0]
    raise FileNotFoundError(
        f"Giải nén xong nhưng không thấy .jpg ở '{extract_to}' hay thư mục con của nó. "
        f"Nội dung: {glob.glob(os.path.join(extract_to, '**'), recursive=True)[:20]}"
    )

## 1. helmet_review — With Helmet / Without Helmet

Chạy cell dưới, bấm **Choose Files**, chọn zip của `datasets/cvat_review/helmet_review`.

In [ ]:
helmet_dir = upload_and_unzip("helmet_review")
build_coco(
    helmet_dir,
    prompt_to_label={
        # Prompt phải nhắm thẳng vào VẬT THỂ (cái mũ), không mô tả qua người —
        # "person wearing a helmet" khiến SAM3 khoanh CẢ NGƯỜI thay vì chỉ cái mũ
        # (đã dính lỗi này thật, box ra to trùm hết cả thân người).
        "safety helmet": "With Helmet",
        "bare human head, no helmet": "Without Helmet",
    },
    out_path="/content/helmet_review_sam3_coco.json",
)

## 2. plate_review — plate (No Plate không auto-label được, để trống tự đánh tay)

Chạy cell dưới, bấm **Choose Files**, chọn zip của `datasets/cvat_review/plate_review`.

In [ ]:
plate_dir = upload_and_unzip("plate_review")
build_coco(
    plate_dir,
    prompt_to_label={
        "vehicle license plate": "plate",
    },
    out_path="/content/plate_review_sam3_coco.json",
)

## 3. new_classes_pool — Mirror

Chạy cell dưới, bấm **Choose Files**, chọn zip của `datasets/cvat_review/new_classes_pool`.

(Đã bỏ `Leg Riding`/`Leg Standing` khỏi notebook này — dự án dùng thẳng pose model có sẵn `yolov8n-pose` + tính góc hông-gối-mắt cá trong `app/cv/pose.py` để phân biệt riding/standing, đã test đúng trên video thật, không cần train thêm detector riêng cho phần này. Box tĩnh không thể hiện được góc khớp như skeleton, nên bỏ hướng này.)

In [ ]:
newclass_dir = upload_and_unzip("new_classes_pool")
build_coco(
    newclass_dir,
    prompt_to_label={
        "motorcycle side mirror": "Mirror",
    },
    out_path="/content/new_classes_pool_sam3_coco.json",
)

## 4. Tải 3 file kết quả về máy và import vào CVAT

In [ ]:
from google.colab import files

files.download("/content/helmet_review_sam3_coco.json")
files.download("/content/plate_review_sam3_coco.json")
files.download("/content/new_classes_pool_sam3_coco.json")

Trong CVAT, mở đúng task tương ứng → menu 3 chấm → **Upload annotations** → format **COCO 1.0** → chọn file JSON vừa tải:
- `helmet_review_sam3_coco.json` → task `helmet_review_v1`
- `plate_review_sam3_coco.json` → task `plate_review_v1`
- `new_classes_pool_sam3_coco.json` → task `new_classes_v1`

Ảnh trong task PHẢI trùng tên với `file_name` trong JSON (đã khớp sẵn vì cùng lấy từ `datasets/cvat_review/`). Sau khi import, CVAT sẽ hiện sẵn các box SAM3 đã đoán — bạn chỉ cần xoá box sai, sửa box lệch, và **đổi `Leg Riding` → `Leg Standing`** cho những trường hợp thực tế là dắt bộ.